## Should this Marvel Character have Blue Eyes?

#### import libraries

In [ ]:
import pandas as pd
import seaborn as sns
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
import sklearn.linear_model as linear_model
from sklearn.metrics import mean_squared_error
from sklearn.metrics import precision_recall_fscore_support


### Data Cleaning and Organizing

In [ ]:
df = pd.read_csv('../input/marvel-comics-character-dataset - marvel-wikia-data.csv')
df.head()

In [ ]:
df[df['eyes'] == 'blue'].head()

In [ ]:
len(df[df['eyes'].isnull()])

In [ ]:
len(df[df['eyes'].notnull()])

In [ ]:
df.groupby(['eyes'])['id'].count()

In [ ]:
# code for blue and not blue

In [ ]:
df['blue_eyes']= df['eyes']

In [ ]:
df['blue_eyes'] = df['blue_eyes'].where(df['blue_eyes'].notnull(), -1)

In [ ]:
prediction_df = df[df['blue_eyes'] == -1].copy()
prediction_df.head()

In [ ]:
# Checking for null values below in order to organize the data. 

In [ ]:
prediction_df.info()

In [ ]:
prediction_df.drop(columns = ['eyes', 'blue_eyes'], inplace=True)

In [ ]:
import random
id_ = random.sample(set(prediction_df.index),1)[0]
data_ = prediction_df.loc[id_]

In [ ]:
id_

In [ ]:
data_.to_dict()

In [ ]:
name = data_.pop('name')

In [ ]:
name

In [ ]:
prediction_df.head()

In [ ]:
prediction_df.to_csv('prediction_data.csv')

In [ ]:
df.drop(prediction_df.index, inplace=True)

In [ ]:
## continue to check for null values. I decide that I'll replace null with -1 to signify incomplete data

In [ ]:
df.info()

In [ ]:
## I one-hot encode blues eyes as 0, not blue as 1 and then drop text. 

In [ ]:
df['blue_eyes'].where(df['blue_eyes'] == 'blue', 0, inplace=True)

In [ ]:
df['blue_eyes'].where(df['blue_eyes'] != 'blue', 1, inplace=True)

In [ ]:
df.head()

In [ ]:
df[df['blue_eyes'] == 1].head()

In [ ]:
df.drop(columns=['eyes'], inplace=True)

#### One Hot encoding of Features

In [ ]:
# distributions and sampling

In [ ]:
df['identity'].unique()

In [ ]:
df.columns

In [ ]:
columns_to_encode = ['identity', 'alignment', 'orientation', 'status',
        'introduced_month',  'hair',
       'gender', ]
         

In [ ]:
def encode_map_dict(names):
    counts_map = [(y, x+1) for x, y in enumerate(names) ]
    co_maps = dict(counts_map)
    return(co_maps)

def make_unique_sets_encoding(column, df_f):
    list_feature_list=df_f[column].unique()
    diction_ = encode_map_dict(list_feature_list)
    return encode_map_dict(list_feature_list)

def encode_columns(df_f, columns):
    for x in columns:
        x_name = str(x) + '_num'
        dict_x = make_unique_sets_encoding(x, df_f)
        df_f[x_name]= df_f[x].map(dict_x);
        
        
encode_columns(df, columns_to_encode)


In [ ]:
# Predictions work flow
encode_columns(prediction_df, columns_to_encode)
prediction_df.drop(columns=columns_to_encode, inplace=True)

In [ ]:
# more cleaning of null values after encoding. 

In [ ]:
prediction_df.info()

In [ ]:
prediction_df['appearances'] = prediction_df['appearances'].where(prediction_df['appearances'].notnull(), -1)
prediction_df['introduced'] = prediction_df['introduced'].where(prediction_df['introduced'].notnull(), -1)
prediction_df.info()

In [ ]:
df.drop(columns=columns_to_encode, inplace=True)

In [ ]:
df.head()

In [ ]:
len(set(df['name']))

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
### Getting rid of null values

In [ ]:
df['appearances'] = df['appearances'].where(df['appearances'].notnull(), -1)

In [ ]:
df['introduced'] = df['introduced'].where(df['introduced'].notnull(), -1)

In [ ]:
# no null values!

In [ ]:
df.info()

In [ ]:
names_comics = df['name']
df.drop(columns=['name'], inplace=True)

In [ ]:
df.to_csv('blue_eyes_data.csv')

## Training and Testing

In [ ]:
# correlation matrices

In [ ]:
fig1, ax = plt.subplots(1,1, figsize=(5,4))
corr = df.corr().round(2)
ax = sns.heatmap(corr, annot=True,cmap='Reds',
            xticklabels=corr.columns.values,
            yticklabels=corr.columns.values,ax = ax )

plt.savefig('correlationMatrix')

In [ ]:
import random
random.seed(42)

In [ ]:
# pick 1962 random samples from NON-Blue entries

In [ ]:
random_index = random.sample(set(df[df['blue_eyes']==0].index), 1962)

In [ ]:
blue_eyes_index = df[df['blue_eyes']==1].index

In [ ]:
training_data = df.loc[random_index].append(df.loc[blue_eyes_index])

In [ ]:
target_variable = training_data['blue_eyes']
training_data.drop(columns=['blue_eyes'], inplace=True)

In [ ]:
feat_X_train, X_test, feat_y_train,y_test = train_test_split(training_data, target_variable, 
                                                             test_size=0.33, random_state=42)

In [ ]:
# baseline ML algorithm

In [ ]:
blues_base = y_test.copy()

In [ ]:
# supposedly only 17% of people in the world have blue eyes, so we random say %17 of testing has blue eyes
blues_test = random.sample(set(y_test.index), int(len(y_test)*.17))

In [ ]:
blues_base.loc[blues_test] = 1

In [ ]:
blues_base.loc[~blues_base.isin(blues_test)] = 0

In [ ]:
mean_squared_error( y_test, blues_base,)

In [ ]:
mean_sq_er_base = mean_squared_error( y_test, blues_base,)

In [ ]:
blues_base.unique()

In [ ]:
y_test = y_test.astype(int)

In [ ]:
precision_recall_fscore_support(y_test,blues_base)

In [ ]:
from sklearn.metrics import f1_score
f1_score(y_test, blues_base, labels=np.unique(blues_base))

In [ ]:
len(blues_base[blues_base == y_test])/(len(blues_base))

In [ ]:
len(y_test[blues_base == y_test])/(len(y_test))

### Logistic Regression

In [ ]:
feat_y_train = feat_y_train.astype(int)
clf = LogisticRegression(random_state=0, solver='newton-cg').fit(feat_X_train, feat_y_train);


In [ ]:
blue_pred = clf.predict(X_test);

In [ ]:
len(blue_pred[blue_pred==y_test])/len(blue_pred)

In [ ]:
mean_squared_error(blue_pred, y_test)

In [ ]:
mean_squared_error(y_test, blue_pred)

In [ ]:
precision_recall_fscore_support(y_test, blue_pred)

In [ ]:
clf.score(X_test, y_test)

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(feat_X_train, feat_y_train)

blue_pred_gnb = gnb.predict(X_test)

precision_recall_fscore_support(y_test, blue_pred_gnb)

In [ ]:
mean_squared_error(blue_pred_gnb, y_test)

In [ ]:
len(blue_pred_gnb[blue_pred_gnb == y_test])/(len(blue_pred_gnb))

## RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(n_estimators=100, max_depth=3,
                             random_state=0)
model.fit(feat_X_train, feat_y_train)  
blue_pred = clf.predict(X_test);

In [ ]:
precision_recall_fscore_support(y_test, blue_pred_gnb)

In [ ]:
mean_squared_error(blue_pred, y_test)

In [ ]:
len(blue_pred[blue_pred == y_test])/(len(blue_pred))

In [ ]:
# test various ML for best RSME or ROC_AUC 

# pickle the model

In [ ]:
# Random Forest model
import pickle

filename = 'finalized_model.sav'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
# pick best ML algorithm and code for deployment

In [ ]:
# flask and dockerize code for a fun web app or combine with other code for a fun API